In [1]:
# imports
import pandas as pd
import numpy as np
import xgboost as xgb
from xgboost import XGBClassifier
import shap

c:\Users\will6\miniconda3\envs\cs320\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#### BASE

#tested various hyperparameters

#read csv
training = pd.read_csv("train_data_w.csv")

#features
features = ['seed_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]


#Initialize arrays
training_dfs = []
base_val_dfs = []
val_years = []
years = [2011,2012,2013,2014,2015,2016,2017,2018,2019,2021,2022,2023,2024]

for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    temp_training = training.query("Season <= @i")
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    base_val_dfs.append(temp_val)
    val_years.append(val_year)

### Model

base_val_error = []

#model
base_mod = XGBClassifier(n_estimators=600, max_depth=2, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 6, max_bin = 20, min_child_weight = 2, num_parallel_tree = 1, objective='binary:logistic', seed = 323)

for i in range(len(years)):
    X_train = training_dfs[i][features]
    y_train = training_dfs[i]['result']

    val = base_val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    base_mod.fit(X_train, y_train)
    
    #make predictions
    preds = base_mod.predict_proba(X_val)
    val['pred'] = preds[:,1]
    val['loss'] = (val['pred'] - val['result'])**2

    base_val_dfs[i] = val.copy()
    print(val_years[i], "Loss:", np.mean(val['loss']))
    base_val_error.append(np.mean(val['loss']))

print("Last 5 Loss:", np.mean(base_val_error[-5:]))

#print(val_base.drop('loss', axis = 1).sort_values('pred', ascending = False).head(5))
#val_base.drop('loss', axis = 1).sort_values('pred', ascending = True).head(5)

#shap
explainer = shap.TreeExplainer(base_mod)
shap_values = explainer.shap_values(X_val)
#shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=50)


2012 Loss: 0.1252367294825085
2013 Loss: 0.1634593735090288
2014 Loss: 0.14039321775698474
2015 Loss: 0.12279191141420114
2016 Loss: 0.1750416652835347
2017 Loss: 0.15185011710010235
2018 Loss: 0.16398571643455837
2019 Loss: 0.13240245756508365
2021 Loss: 0.14583036366358554
2022 Loss: 0.1606158278621548
2023 Loss: 0.16352126026940866
2024 Loss: 0.12013448296193462
2025 Loss: 0.11288653362987944
Last 5 Loss: 0.1405976936773926


In [3]:
#Rolling

#read csv
training = pd.read_csv("train_data_w.csv")

#features
features = ['seed_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]


#Initialize arrays
training_dfs = []
r_val_dfs = []
val_years = []
years = [2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2021,2022,2023,2024]

y_count = 8

for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    if i <= 2019 or i >= 2020 + y_count:
        seasons = [i - k for k in range(y_count)]
    else:
        seasons = [i - k for k in range(y_count+1)]

    temp_training = training.query("Season in @seasons")
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    r_val_dfs.append(temp_val)
    val_years.append(val_year)

### Model

#model
r_mod = XGBClassifier(n_estimators=500, max_depth=2, learning_rate=0.015, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 4, max_bin = 20, min_child_weight = 4, num_parallel_tree = 1, objective='binary:logistic', seed = 323)

r_val_error = []

for i in range(len(years)):
    X_train = training_dfs[i][features]
    y_train = training_dfs[i]['result']

    val = r_val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    r_mod.fit(X_train, y_train)
    
    #make predictions
    preds = r_mod.predict_proba(X_val)
    val['pred'] = preds[:,1]
    val['loss'] = (val['pred'] - val['result'])**2

    r_val_dfs[i] = val.copy()
    r_val_error.append(np.mean(val['loss']))
    print(val_years[i], "Loss:", np.mean(val['loss']), " |  Dif:", np.mean(val['loss']) - base_val_error[val_years[i] - 2017])

print("Last 5 Loss:", np.mean(r_val_error[-5:]), " | Dif:", np.mean(r_val_error[-5:]) - np.mean(base_val_error[-5:]))

#shap
explainer = shap.TreeExplainer(r_mod)
shap_values = explainer.shap_values(X_val)
#shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=50)


2011 Loss: 0.14659917305421122  |  Dif: 0.01419671548912757
2012 Loss: 0.1261860376637552  |  Dif: -0.019644325999830348
2013 Loss: 0.17066094866217368  |  Dif: 0.010045120800018875
2014 Loss: 0.14145280883284878  |  Dif: -0.02206845143655989
2015 Loss: 0.11498867537984972  |  Dif: -0.005145807582084896
2016 Loss: 0.18514475270191766  |  Dif: 0.07225821907203822
2017 Loss: 0.15775391941145414  |  Dif: 0.032517189928945645
2018 Loss: 0.16367915432274  |  Dif: 0.00021978081371118452
2019 Loss: 0.13697482699026528  |  Dif: -0.003418390766719459
2021 Loss: 0.1447539662537561  |  Dif: -0.030287699029778598
2022 Loss: 0.1622442712430923  |  Dif: 0.010394154142989942
2023 Loss: 0.16892124603730915  |  Dif: 0.004935529602750788
2024 Loss: 0.13159272293729934  |  Dif: -0.0008097346277843032
2025 Loss: 0.10553073482655781  |  Dif: -0.04029962883702773
Last 5 Loss: 0.14260858825960293  | Dif: 0.002010894582210321


In [4]:
#### Weights

#read csv
training = pd.read_csv("train_data_w.csv")

#features
features = ['seed_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]

#Initialize arrays
training_dfs = []
w_val_dfs = []
val_years = []
years = [2011,2012,2013,2014,2015,2016,2017,2018,2019,2021,2022,2023,2024]

#Weight parameter
weight_param = 0.95

for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    temp_training = training.query("Season <= @i")
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    w_val_dfs.append(temp_val)
    val_years.append(val_year)

### Model

#model
w_mod = XGBClassifier(n_estimators=500, max_depth=2, learning_rate=0.015, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 6, max_bin = 20, min_child_weight = 2, num_parallel_tree = 1, objective='binary:logistic', seed = 323)

w_val_error = []

for i in range(len(years)):
    temp_df = training_dfs[i]
    temp_df = temp_df.assign(weight = (weight_param ** (temp_df['Season'].max() - temp_df['Season'])).clip(0.1))

    X_train = temp_df[features]
    y_train = temp_df['result']
    train_weights = temp_df['weight']

    val = w_val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    w_mod.fit(X_train, y_train, sample_weight=train_weights)
    
    #make predictions
    preds = w_mod.predict_proba(X_val)
    val['pred'] = preds[:,1]
    val['loss'] = (val['pred'] - val['result'])**2

    w_val_dfs[i] = val.copy()
    w_val_error.append(np.mean(val['loss']))
    print(val_years[i], "Loss:", np.mean(val['loss']), " |  Dif:", np.mean(val['loss']) - base_val_error[val_years[i] - 2017])

print("Last 5 Loss:", np.mean(w_val_error[-5:]), " | Dif:", np.mean(w_val_error[-5:]) - np.mean(base_val_error[-5:]))

#shap
explainer = shap.TreeExplainer(w_mod)
shap_values = explainer.shap_values(X_val)
#shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=50)


2012 Loss: 0.1246030127863913  |  Dif: -0.02122735087719424
2013 Loss: 0.16368688505183518  |  Dif: 0.0030710571896803707
2014 Loss: 0.14179024692480277  |  Dif: -0.021731013344605893
2015 Loss: 0.12175425902218485  |  Dif: 0.0016197760602502387
2016 Loss: 0.1792142098287499  |  Dif: 0.06632767619887045
2017 Loss: 0.15360575505184307  |  Dif: 0.02836902556933457
2018 Loss: 0.1629469789254199  |  Dif: -0.0005123945836089105
2019 Loss: 0.13081875916014382  |  Dif: -0.009574458596840918
2021 Loss: 0.14350877481821925  |  Dif: -0.03153289046531546
2022 Loss: 0.16233306103599413  |  Dif: 0.010482943935891775
2023 Loss: 0.16463109309557292  |  Dif: 0.0006453766610145517
2024 Loss: 0.12454362446292407  |  Dif: -0.007858833102159582
2025 Loss: 0.11288695453202138  |  Dif: -0.032943409131564155
Last 5 Loss: 0.14158070158894637  | Dif: 0.000983007911553757


In [5]:
### Combine

#Merge Data
def get_vals(lst, prefix):
    df = pd.concat(lst, ignore_index=True)[['Season', 'DayNum', 'team_A', 'team_B', 'score_A', 'score_B', 'result', 'pred', 'loss']]
    df = df.rename({'pred': f'{prefix}pred', 'loss': f'{prefix}loss'}, axis='columns')
    return df

base_val = get_vals(base_val_dfs, "base_")
r_val = get_vals(r_val_dfs, "r_")
w_val = get_vals(w_val_dfs, "w_")

merge_keys = ["Season", 'DayNum', 'team_A', 'team_B', 'score_A', 'score_B', 'result']

full_val = pd.merge(base_val, r_val, how="outer", on=merge_keys).merge(
        w_val, how="outer", on=merge_keys)


full_val.tail()


#combined prediction
base_weight = 0.4
r_weight = 0.3
w_weight = 0.3

full_val = full_val.assign(com_pred = full_val['base_pred']*base_weight + full_val['r_pred']*r_weight + full_val['w_pred']*w_weight)
full_val['com_loss'] = (full_val['com_pred'] - full_val['result'])**2

full_val.tail()

#group and summarize
val_summary = full_val.groupby(["Season"]).agg(
    base_val=("base_loss", "mean"),
    r_val=("r_loss", "mean"),
    w_val=("w_loss", "mean"),
    com_val=("com_loss", "mean")).reset_index()

val_summary.loc['last5'] = val_summary.set_index('Season').iloc[-5:].mean()

val_summary.tail(6)

,Season,base_val,r_val,w_val,com_val
9,2021.0,0.145830,0.144754,0.143509,0.144403
10,2022.0,0.160616,0.162244,0.162333,0.161291
11,2023.0,0.163521,0.168921,0.164631,0.164870
12,2024.0,0.120134,0.131593,0.124544,0.124478
13,2025.0,0.112887,0.105531,0.112887,0.110040
last5,NaN,0.140598,0.142609,0.141581,0.141016


In [6]:
data = pd.read_csv("data_for_submission_w.csv")

#read csv
training = pd.read_csv("train_data_w.csv")

#features
features = ['seed_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]

base_training = training.query("Season <= 2025")
r_training = training.query("Season <= 2025").query("Season >= 2017")
w_training = training.query("Season <= 2025")

#model
base_mod = XGBClassifier(n_estimators=600, max_depth=2, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 6, max_bin = 20, min_child_weight = 2, num_parallel_tree = 1, objective='binary:logistic', seed = 323)
r_mod = XGBClassifier(n_estimators=500, max_depth=2, learning_rate=0.015, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 4, max_bin = 20, min_child_weight = 4, num_parallel_tree = 1, objective='binary:logistic', seed = 323)
w_mod = XGBClassifier(n_estimators=500, max_depth=2, learning_rate=0.015, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 6, max_bin = 20, min_child_weight = 2, num_parallel_tree = 1, objective='binary:logistic', seed = 323)

#training data
X_train_base = base_training[features]
y_train_base = base_training['result']

X_train_r = r_training[features]
y_train_r = r_training['result']

w_training = w_training.assign(weight = (0.95 ** (w_training['Season'].max() - w_training['Season'])).clip(0.1))
X_train_w = w_training[features]
y_train_w = w_training['result']
train_weights = w_training['weight']

#fit model
base_mod.fit(X_train_base, y_train_base)
r_mod.fit(X_train_r, y_train_r)
w_mod.fit(X_train_w, y_train_w, sample_weight=train_weights)
    
#data for making preds
X_data_base = data[features]
X_data_r = data[features]
X_data_w = data[features]

base_preds = base_mod.predict_proba(X_data_base)
r_preds = r_mod.predict_proba(X_data_r)
w_preds = w_mod.predict_proba(X_data_w)

#make predictions
base_preds = base_mod.predict_proba(X_data_base)
r_preds = r_mod.predict_proba(X_data_r)
w_preds = w_mod.predict_proba(X_data_w)

#put preds in df
data['base_pred'] = base_preds[:,1]
data['r_pred'] = r_preds[:,1]
data['w_pred'] = w_preds[:,1]

data = data.assign(com_pred = data['base_pred']*0.4 + data['r_pred']*0.3 + data['w_pred']*0.3)


In [7]:
com_submission_file = data.assign(
        ID="2026_" + data['team_A'].astype(str) + "_" + data['team_B'].astype(str).copy()
    ).assign(Pred = data['com_pred'])[['ID', 'Pred']]

base_submission_file = data.assign(
        ID="2026_" + data['team_A'].astype(str) + "_" + data['team_B'].astype(str).copy()
    ).assign(Pred = data['base_pred'])[['ID', 'Pred']]

r_submission_file = data.assign(
        ID="2026_" + data['team_A'].astype(str) + "_" + data['team_B'].astype(str).copy()
    ).assign(Pred = data['r_pred'])[['ID', 'Pred']]

w_submission_file = data.assign(
        ID="2026_" + data['team_A'].astype(str) + "_" + data['team_B'].astype(str).copy()
    ).assign(Pred = data['w_pred'])[['ID', 'Pred']]



com_submission_file.to_csv("submission_files/women_com_submission_file.csv", index=False)
base_submission_file.to_csv("submission_files/women_base_submission_file.csv", index=False)
r_submission_file.to_csv("submission_files/women_r_submission_file.csv", index=False)
w_submission_file.to_csv("submission_files/women_w_submission_file.csv", index=False)

In [8]:
### Main Submission
main_submission = data.assign(
        ID="2026_" + data['team_A'].astype(str) + "_" + data['team_B'].astype(str).copy()
    ).assign(Pred = data['com_pred'])

main_submission.loc[main_submission['seed_dif'] >= 9, 'Pred'] = 0
main_submission.loc[main_submission['seed_dif'] <= -9, 'Pred'] = 1

main_submission = main_submission[['ID', 'Pred']]

main_submission.to_csv("submission_files/women_main_submission_file.csv", index=False)